# Agentes RAG sobre el corpus legal + evaluación con el juez

Corre varias variantes del sistema sobre `sample_50.jsonl`, guarda cada entrega y las mide dos veces:
1. **Métricas locales** (sin llave, gratis): exactitud en cerradas, calidad de citación aproximada, abstenciones, formato y segundos por pregunta.
2. **Evaluador oficial** (`evaluate.py --ragas`) con la llave del juez de OpenRouter. Solo se usa aquí, para medir: el sistema nunca la toca.

Todo es abierto: decoder GGUF en llama.cpp (Qwen3, temperatura 0), bge-m3 + BM25 para recuperar.
Corre en **Colab**, **Kaggle** o un **PC con GPU de 8 GB**. **Guarda el avance**: cada respuesta se escribe al momento; si se cae, vuelve a correr todo y sigue donde iba.

## Antes de correr

En la misma carpeta de siempre (`MyDrive/hackathon_vectorial`, la del notebook de vectores) tiene que haber:
- lo de `kit_agentes.zip` (los `.py`, incluido el nuevo `rag.py`, y este notebook),
- `corpus.db` o `corpus_v1.zip`, y la carpeta `indices/` que dejó el notebook de vectores (o `vectores_bge-m3.zip`),
- `sample_50.jsonl`,
- el **material del reto** tal como lo entregaron (la carpeta `reto/`, con `scripts/evaluate.py`, `scripts/citations.py`, `schema/` y `data/`). `rag.py` usa `citations.py` para verificar las citas igual que el evaluador.

**La llave del juez** no se escribe en ningún archivo. En Colab: ícono de la llave (Secrets) a la izquierda → *Add new secret* → nombre `OPENROUTER_API_KEY`, pegar la llave y activar *Notebook access*. En Kaggle: *Add-ons → Secrets* con el mismo nombre. Si no está, el notebook la pide con un cuadro oculto.

**Colab:** GPU T4, luego *Ejecutar todo*. La primera vez compila llama.cpp con CUDA (unos 20 min) y guarda la rueda en Drive; las siguientes veces se instala en segundos.

In [ ]:
# ===== Configuracion =====
# Modelos GGUF abiertos (<= 8B). Se baja el primer archivo del repo que contenga el patron.
MODELOS = {
    "qwen3-8b":      ("Qwen/Qwen3-8B-GGUF", "Q4_K_M"),                    # Apache 2.0, ~5 GB
    "qwen3-4b-2507": ("unsloth/Qwen3-4B-Instruct-2507-GGUF", "Q4_K_M"),   # Apache 2.0, ~2,5 GB, mas rapido
}
# Variantes a comparar. variante: "simple" (1 busqueda, 1 generacion) o "agente" (planea consultas,
# busca con todas, y si cito algo que no estaba trae ese articulo y regenera una vez).
VARIANTES = {
    "simple_8b_k8": dict(modelo="qwen3-8b", variante="simple", k=8),
    "agente_8b_k8": dict(modelo="qwen3-8b", variante="agente", k=8),
    "simple_8b_k5": dict(modelo="qwen3-8b", variante="simple", k=5),
    "simple_4b_k8": dict(modelo="qwen3-4b-2507", variante="simple", k=8),
}
CORRER = list(VARIANTES)        # o solo algunas: ["simple_8b_k8", "agente_8b_k8"]
SOLO_PRIMERAS = None            # p. ej. 5 para una prueba rapida (el evaluador oficial espera las 50)
N_CTX = 8192                    # tokens de contexto del decoder (con 8 GB de VRAM, 6144 si falta memoria)
ABSTENCION = "modelo"           # "modelo" o "nunca"
USAR_JUEZ = True                # correr evaluate.py --ragas (gasta la llave; los resultados quedan guardados y no se repiten)
LLAMA_CPP_VERSION = "0.3.35"
CARPETA = None                  # None = automatico (MyDrive/hackathon_vectorial, /kaggle/working/..., o esta carpeta)

In [ ]:
# ===== Entorno y carpeta de trabajo =====
import os
os.environ["JAX_PLATFORMS"] = "cpu"  # bm25s usa JAX si esta instalado (Colab) y JAX se apodera del 75 % de la GPU
import sys, glob, json, shutil, sqlite3, subprocess, time, zipfile
from pathlib import Path

try:
    import google.colab  # noqa
    ENTORNO = "colab"
except ImportError:
    ENTORNO = "kaggle" if os.path.exists("/kaggle/input") else "local"

if ENTORNO == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path(CARPETA or "/content/drive/MyDrive/hackathon_vectorial")
    RAPIDO = Path("/content/rapido")
    BUSCAR_EN = [WORK, Path("/content")]
elif ENTORNO == "kaggle":
    WORK = Path(CARPETA or "/kaggle/working/hackathon_vectorial")
    RAPIDO = Path("/tmp/rapido")
    BUSCAR_EN = [WORK, Path("/kaggle/input")]
else:
    WORK = Path(CARPETA or Path.cwd())
    RAPIDO = WORK / "data"
    BUSCAR_EN = [WORK]
WORK.mkdir(parents=True, exist_ok=True)
RAPIDO.mkdir(parents=True, exist_ok=True)
ENTREGAS = WORK / "entregas"
ENTREGAS.mkdir(exist_ok=True)
os.environ["PYTHONIOENCODING"] = "utf-8"
print(f"entorno: {ENTORNO}\ncarpeta de trabajo (aqui queda el avance): {WORK}")

In [ ]:
# ===== Dependencias (llama.cpp con CUDA se compila una vez y queda guardado) =====
import importlib.util
faltan = [p for m, p in [("bm25s", "bm25s"), ("faiss", "faiss-cpu"), ("sentence_transformers", "sentence-transformers"),
                         ("jinja2", "jinja2"), ("huggingface_hub", "huggingface_hub"), ("pandas", "pandas")]
          if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

import torch
assert torch.cuda.is_available(), "No hay GPU: en Colab/Kaggle activa la GPU; en un PC instala torch con CUDA."
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB)")

def llama_con_gpu():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp, sys; sys.exit(0 if llama_cpp.llama_supports_gpu_offload() else 1)"],
                       capture_output=True)
    return r.returncode == 0

if not llama_con_gpu():
    if ENTORNO == "local":
        raise SystemExit("Instala llama-cpp-python con CUDA (ver su README: CMAKE_ARGS=-DGGML_CUDA=on) y vuelve a correr.")
    cc = "".join(map(str, torch.cuda.get_device_capability(0)))
    RUEDAS = WORK / "wheels" / f"llama{LLAMA_CPP_VERSION}-sm{cc}-cp{sys.version_info.major}{sys.version_info.minor}"
    ruedas = sorted(RUEDAS.glob("llama_cpp_python-*.whl"))
    if not ruedas:
        print("Compilando llama-cpp-python con CUDA para esta GPU. Es una sola vez (unos 20 min); la rueda queda en", RUEDAS)
        RUEDAS.mkdir(parents=True, exist_ok=True)
        tmp = RAPIDO / "rueda_tmp"
        env = {**os.environ, "CMAKE_ARGS": f"-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES={cc}",
               "CMAKE_BUILD_PARALLEL_LEVEL": str(os.cpu_count() or 2)}
        subprocess.check_call([sys.executable, "-m", "pip", "wheel", "--no-deps", "-w", str(tmp),
                               f"llama-cpp-python=={LLAMA_CPP_VERSION}"], env=env)
        for w in tmp.glob("*.whl"):
            shutil.copy(w, RUEDAS / w.name)  # a Drive al final: si se corta, no queda una rueda a medias
        ruedas = sorted(RUEDAS.glob("llama_cpp_python-*.whl"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", str(ruedas[-1])])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diskcache", "typing-extensions"])
    assert llama_con_gpu(), "llama-cpp-python quedo sin soporte de GPU"
print("llama.cpp con GPU: ok")

In [ ]:
# ===== Encontrar codigo, corpus.db, indices, sample_50 y evaluate.py =====
def buscar_archivo(nombre, excluir=("/indices/", "/entregas/", "/wheels/")):
    for raiz in BUSCAR_EN:
        if (raiz / nombre).exists():
            return raiz / nombre
        hallados = [h for h in sorted(glob.glob(str(raiz / "**" / nombre), recursive=True))
                    if not any(x in h.replace("\\", "/") for x in excluir)]
        if hallados:
            return Path(hallados[0])
    return None

p = buscar_archivo("rag.py")
if p is None:
    raise SystemExit(f"No encuentro rag.py en {BUSCAR_EN}. Sube el contenido de kit_agentes.zip.")
CODIGO = p.parent
for f in ["rag.py", "buscar.py", "indexar.py", "citas.py", "seed_match.py"]:
    assert (CODIGO / f).exists(), f"falta {f} junto a rag.py ({CODIGO})"
if str(CODIGO) not in sys.path:
    sys.path.insert(0, str(CODIGO))
print(f"codigo: {CODIGO}")

def db_sana(p):
    try:
        con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
        ok = con.execute("PRAGMA quick_check").fetchone()[0] == "ok"
        con.close()
        return ok
    except sqlite3.Error:
        return False

DB = RAPIDO / "corpus.db"
if ENTORNO == "local" and buscar_archivo("corpus.db") is not None:
    DB = buscar_archivo("corpus.db")
elif not db_sana(DB):
    tmp = RAPIDO / "corpus.db.tmp"
    suelto = buscar_archivo("corpus.db")
    if suelto is not None and db_sana(suelto):
        print(f"copiando {suelto} al disco local...")
        shutil.copy(suelto, tmp)
    else:
        comprimido = buscar_archivo("corpus_v1.zip")
        assert comprimido is not None, "No encuentro corpus.db ni corpus_v1.zip"
        print(f"descomprimiendo {comprimido}...")
        with zipfile.ZipFile(comprimido) as z, z.open(next(n for n in z.namelist() if n.endswith("corpus.db"))) as src, \
                open(tmp, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 24)
    os.replace(tmp, DB)
assert db_sana(DB), f"{DB} esta danada"
print(f"corpus.db: {DB}")

def indice(nombre):
    for raiz in [WORK / "indices", RAPIDO / "indices", *BUSCAR_EN]:
        hall = sorted(glob.glob(str(raiz / "**" / nombre / "dense.faiss"), recursive=True)) or \
               ([str(raiz / nombre / "dense.faiss")] if (raiz / nombre / "dense.faiss").exists() else [])
        if hall:
            return Path(hall[0]).parent
    return None

if indice("index_sin_sentencias") is None or indice("index_juris") is None:
    z = buscar_archivo("vectores_bge-m3.zip")
    assert z is not None, "No encuentro indices/ (del notebook de vectores) ni vectores_bge-m3.zip"
    print(f"descomprimiendo {z}...")
    with zipfile.ZipFile(z) as zz:
        zz.extractall(RAPIDO / "indices")
INDEX, INDEX_JURIS = indice("index_sin_sentencias"), indice("index_juris")
assert (INDEX / "bm25").exists() and (INDEX_JURIS / "bm25").exists(), "los indices no traen bm25/"
print(f"indices: {INDEX} + {INDEX_JURIS}")

MUESTRA = buscar_archivo("sample_50.jsonl")
assert MUESTRA is not None, "falta sample_50.jsonl"
ITEMS = [json.loads(l) for l in MUESTRA.read_text(encoding="utf-8").splitlines() if l.strip()]
if SOLO_PRIMERAS:
    ITEMS = ITEMS[:SOLO_PRIMERAS]
print(f"preguntas: {len(ITEMS)}")

EVALUATE = buscar_archivo("evaluate.py")
print(f"evaluate.py: {EVALUATE or 'NO ESTA: sube la carpeta del material del reto (reto/) para medir con el evaluador oficial'}")
if EVALUATE is not None and str(EVALUATE.parent) not in sys.path:
    sys.path.append(str(EVALUATE.parent))  # rag.py usa scripts/citations.py para verificar citas igual que el evaluador

In [ ]:
# ===== Buscador (hibrido BM25 + bge-m3, sin reranker: no aporto en la medicion) =====
import importlib, rag, buscar
importlib.reload(buscar); importlib.reload(rag)
B = buscar.Buscador(str(DB), str(INDEX), index_juris=str(INDEX_JURIS))
print("buscador listo:", "hibrido" if B.faiss_index is not None else "solo BM25")
print("verificacion de citas:", "extractor oficial (scripts/citations.py)" if rag.oficial() else "citas.py propio (sube el material del reto)")
con = sqlite3.connect(DB)
sin_offsets = con.execute("SELECT COUNT(*) FROM chunks WHERE inicio IS NULL").fetchone()[0]
con.close()
if sin_offsets:
    print(f"OJO: {sin_offsets} fragmentos sin inicio/fin. Para medir da igual (se omiten), pero antes de la entrega "
          "hay que correr exportar_entrega.py --guardar-offsets sobre esta corpus.db.")

In [ ]:
# ===== Correr las variantes (retoma sola) =====
import gc
from huggingface_hub import hf_hub_download, list_repo_files

def bajar_modelo(nombre):
    repo, patron = MODELOS[nombre]
    archivo = next(f for f in sorted(list_repo_files(repo))
                   if f.endswith(".gguf") and patron.lower() in f.lower() and "mmproj" not in f.lower())
    return hf_hub_download(repo, archivo, local_dir=str(RAPIDO / "modelos" / nombre))

LLM_ACTUAL, LLM_NOMBRE = None, None
orden = sorted(CORRER, key=lambda v: VARIANTES[v]["modelo"])  # agrupa por modelo: se carga una vez cada uno
for nombre in orden:
    cfg = VARIANTES[nombre]
    salida = ENTREGAS / f"{nombre}.jsonl"
    hechos = sum(1 for _ in open(salida, encoding="utf-8")) if salida.exists() else 0
    print(f"\n===== {nombre} ({cfg}) =====")
    if hechos >= len(ITEMS):
        print("ya estaba lista")
        continue
    if LLM_NOMBRE != cfg["modelo"]:
        LLM_ACTUAL = None; gc.collect(); torch.cuda.empty_cache()
        ruta = bajar_modelo(cfg["modelo"])
        print(f"cargando {ruta}")
        LLM_ACTUAL, LLM_NOMBRE = rag.LLM(ruta, n_ctx=N_CTX), cfg["modelo"]
    s = rag.Sistema(B, LLM_ACTUAL, cfg["variante"], k=cfg["k"], abstencion=ABSTENCION)
    rag.correr(s, ITEMS, salida)
LLM_ACTUAL = None; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ===== Metricas locales (sin juez) =====
import pandas as pd
filas = {}
for nombre in VARIANTES:
    salida = ENTREGAS / f"{nombre}.jsonl"
    if salida.exists():
        filas[nombre] = rag.metricas_locales(ITEMS, salida)
LOCAL = pd.DataFrame(filas).T
display(LOCAL)
print("calidad_citacion_aprox = recall de las normas de legal_basis (1 si esta en los pasajes, 0,5 si no) "
      "- 2 x tasa de citas ajenas sin respaldo. Es una aproximacion: la oficial la da evaluate.py.\n"
      "Presupuesto del sabado: ~22 s por pregunta.")

In [ ]:
# ===== Evaluador oficial: sin juez siempre (gratis), con juez si USAR_JUEZ (la llave NO se guarda) =====
def llave_juez():
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"]
    try:
        if ENTORNO == "colab":
            from google.colab import userdata
            return userdata.get("OPENROUTER_API_KEY")
        if ENTORNO == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    except Exception as e:
        print(f"(no la encontre en los secretos: {type(e).__name__})")
    import getpass
    return getpass.getpass("Llave del juez (OPENROUTER_API_KEY): ").strip()

def evaluar(nombre, con_juez, env):
    # Corre evaluate.py como en el enunciado (desde la raiz del material) y devuelve el reporte JSON.
    # Con juez, si ya hay un reporte mas nuevo que la entrega, lo reusa: no se gasta la llave dos veces.
    salida = ENTREGAS / f"{nombre}.jsonl"
    rep = ENTREGAS / f"{nombre}.eval{'_ragas' if con_juez else ''}.json"
    if con_juez and rep.exists() and rep.stat().st_mtime > salida.stat().st_mtime:
        return json.loads(rep.read_text(encoding="utf-8"))
    cmd = [sys.executable, str(EVALUATE), "--submission", str(salida), "--split", "sample", "--out", str(rep)]
    r = subprocess.run(cmd + (["--ragas"] if con_juez else []), cwd=EVALUATE.parent.parent, env=env,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    if r.returncode != 0 or not rep.exists():
        print(f"evaluate.py fallo con {nombre}:\n{(r.stderr or r.stdout)[-2000:]}")
        return None
    return json.loads(rep.read_text(encoding="utf-8"))

def fila(rep):
    rg = rep.get("correccion_ragas", {})
    return {"cerradas": f"{rep['cerradas']['aciertos']}/{rep['cerradas']['n']}", "pts_cerradas": rep["cerradas"]["puntos"],
            "ragas": rg.get("correctness"), "pts_ragas": rg.get("puntos"),
            "indice_citas": rep["citas"]["indice"], "citas_sin_respaldo": rep["citas"]["citas_sin_respaldo"],
            "pts_citas": rep["citas"]["puntos"], "pts_abstencion": rep["abstencion"]["puntos"],
            "total": rep["total_automatico"]["obtenidos"], "de": rep["total_automatico"]["posibles"],
            "errores_formato": rep["validacion"]["errores"]}

OFICIAL = {}
if EVALUATE is None:
    print("sin evaluate.py: solo quedan las metricas locales")
else:
    env = {k: v for k, v in os.environ.items() if k != "OPENROUTER_API_KEY"}
    for nombre in VARIANTES:
        if (ENTREGAS / f"{nombre}.jsonl").exists():
            rep = evaluar(nombre, False, env)
            if rep:
                OFICIAL[nombre] = fila(rep)
    if USAR_JUEZ and OFICIAL:
        req = EVALUATE.parent / "requirements-evaluador.txt"
        if req.exists() and not (RAPIDO / ".req_evaluador_ok").exists():
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
            (RAPIDO / ".req_evaluador_ok").touch()
        env_juez = {**env, "OPENROUTER_API_KEY": llave_juez()}
        for nombre in list(OFICIAL):
            print(f"juez: {nombre}...")
            rep = evaluar(nombre, True, env_juez)
            if rep:
                OFICIAL[nombre] = fila(rep)
        del env_juez
    TABLA = pd.DataFrame(OFICIAL).T
    display(TABLA)
    print("total = puntos automaticos obtenidos; 'de' = posibles (50 sin juez, 80 con juez). "
          "Cada reporte completo queda en entregas/<variante>.eval*.json")

In [ ]:
# ===== Resumen para el hilo (se agrega a resultados_agentes.md) =====
linea = f"\n## {time.strftime('%Y-%m-%d %H:%M')} | {ENTORNO} | {torch.cuda.get_device_name(0)}\n\n"
linea += LOCAL.to_markdown() if hasattr(LOCAL, "to_markdown") and importlib.util.find_spec("tabulate") else LOCAL.to_string()
if OFICIAL:
    T = pd.DataFrame(OFICIAL).T
    linea += "\n\n### evaluate.py (oficial)\n\n" + (T.to_markdown() if importlib.util.find_spec("tabulate") else T.to_string())
with open(WORK / "resultados_agentes.md", "a", encoding="utf-8") as f:
    f.write(linea + "\n")
print(linea)
print(f"\n(guardado en {WORK / 'resultados_agentes.md'}; las entregas y trazas estan en {ENTREGAS})")

## Para mirar después

- `entregas/<variante>.jsonl`: la entrega en el formato del anexo A (lo que lee `evaluate.py`).
- `entregas/<variante>.traza.jsonl`: por pregunta, las consultas, el plan del agente, los pasajes, las citas sin respaldo y los segundos. Ahí se ve por qué falló una.
- Para rehacer una variante después de cambiar el código: borra su `.jsonl` y su `.traza.jsonl`.
- Una variante nueva es una línea más en `VARIANTES`.